# Classroom Attention Monitor

This notebook implements a real-time classroom attention monitor using **MediaPipe FaceMesh** and a **TensorFlow CNN**. 

### Features:
- **Drowsiness Detection**: Using Eye Aspect Ratio (EAR).
- **Yawning Detection**: Using Mouth Aspect Ratio (MAR).
- **Gaze Tracking**: Monitoring iris position.
- **Head Pose**: Detecting if students are looking away.
- **CNN Classification**: A secondary check for attentive vs. distracted behavior.

In [1]:
# 1. Setup Dependencies
!pip install tensorflow==2.15.0 mediapipe==0.10.9 opencv-python numpy<2.0.0

zsh:1: no such file or directory: 2.0.0


In [2]:
import cv2
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import load_model
import mediapipe as mp
from collections import deque
import math
import os

# ==================== Configuration ====================
class Config:
    MODEL_PATH = 'models/attention_cnn.h5'
    IMG_SIZE = (128, 128)
    
    # Face Mesh Config
    MAX_FACES = 5 # Reduced for smoother notebook performance
    MIN_DETECTION_CONFIDENCE = 0.5
    MIN_TRACKING_CONFIDENCE = 0.5
    
    # Drowsiness Config
    EAR_THRESHOLD = 0.15
    MAR_THRESHOLD = 0.8
    
    # Visuals
    FONT = cv2.FONT_HERSHEY_SIMPLEX
    SMOOTHING_FRAMES = 5

# Landmarks indices for EAR/MAR/Gaze
RIGHT_EYE = [33, 133, 160, 144, 159, 145, 158, 153] 
LEFT_EYE = [263, 362, 387, 373, 386, 374, 385, 380]
MOUTH = [61, 291, 39, 181, 0, 17, 269, 405]
LEFT_IRIS = [474, 475, 476, 477]
RIGHT_IRIS = [469, 470, 471, 472]

2026-04-03 11:06:27.316457: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2026-04-03 11:06:27.319919: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2026-04-03 11:06:27.330695: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1775194587.349347   43705 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775194587.353962   43705 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1775194587.365837   43705 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linkin

In [3]:
# ==================== Utilities ====================
class PredictionSmoother:
    def __init__(self, size=Config.SMOOTHING_FRAMES):
        self.history = {}
        self.size = size

    def smooth(self, face_id, prediction):
        if face_id not in self.history:
            self.history[face_id] = deque(maxlen=self.size)
        self.history[face_id].append(prediction)
        return sum(self.history[face_id]) / len(self.history[face_id])

def distance(p1, p2):
    return math.sqrt((p1[0] - p2[0])**2 + (p1[1] - p2[1])**2)

def calculate_ear(landmarks, eye_indices, w, h):
    lms = landmarks.landmark
    p = lambda i: (lms[i].x * w, lms[i].y * h)
    v1 = distance(p(eye_indices[2]), p(eye_indices[3]))
    v2 = distance(p(eye_indices[4]), p(eye_indices[5]))
    v3 = distance(p(eye_indices[6]), p(eye_indices[7]))
    horiz = distance(p(eye_indices[0]), p(eye_indices[1]))
    return (v1 + v2 + v3) / (3.0 * horiz)

def calculate_mar(landmarks, mouth_indices, w, h):
    lms = landmarks.landmark
    p = lambda i: (lms[i].x * w, lms[i].y * h)
    v1 = distance(p(mouth_indices[2]), p(mouth_indices[3]))
    v2 = distance(p(mouth_indices[4]), p(mouth_indices[5]))
    v3 = distance(p(mouth_indices[6]), p(mouth_indices[7]))
    horiz = distance(p(mouth_indices[0]), p(mouth_indices[1]))
    return (v1 + v2 + v3) / (3.0 * horiz)

def calculate_gaze(landmarks, eye_indices, iris_indices, w, h):
    lms = landmarks.landmark
    p = lambda i: (lms[i].x * w, lms[i].y * h)
    eye_center_x = (p(eye_indices[0])[0] + p(eye_indices[1])[0]) / 2
    iris_center_x = sum([p(i)[0] for i in iris_indices]) / 4
    eye_width = distance(p(eye_indices[0]), p(eye_indices[1]))
    if eye_width == 0: return 0
    return (iris_center_x - eye_center_x) / eye_width

def get_face_bbox(landmarks, frame_w, frame_h):
    x_coords = [lm.x for lm in landmarks.landmark]
    y_coords = [lm.y for lm in landmarks.landmark]
    xmin, xmax = min(x_coords), max(x_coords)
    ymin, ymax = min(y_coords), max(y_coords)
    w, h = xmax - xmin, ymax - ymin
    xmin, xmax = max(0, xmin - w * 0.1), min(1, xmax + w * 0.1)
    ymin, ymax = max(0, ymin - h * 0.2), min(1, ymax + h * 0.1)
    return (int(xmin * frame_w), int(ymin * frame_h), 
            int((xmax - xmin) * frame_w), int((ymax - ymin) * frame_h))

In [4]:
# ==================== Model Loading ====================
print(f"Loading Model: {Config.MODEL_PATH}...")
model = load_model(Config.MODEL_PATH)
print("✓ Model loaded")

mp_face_mesh = mp.solutions.face_mesh
face_mesh = mp_face_mesh.FaceMesh(
    max_num_faces=Config.MAX_FACES,
    min_detection_confidence=Config.MIN_DETECTION_CONFIDENCE,
    min_tracking_confidence=Config.MIN_TRACKING_CONFIDENCE,
    refine_landmarks=True
)
mp_drawing = mp.solutions.drawing_utils
drawing_spec = mp_drawing.DrawingSpec(thickness=1, circle_radius=1)
smoother = PredictionSmoother()

2026-04-03 11:06:30.599031: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Loading Model: models/attention_cnn.h5...
✓ Model loaded


I0000 00:00:1775194590.784687   43705 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1775194590.789426   43807 gl_context.cc:344] GL version: 3.2 (OpenGL ES 3.2 Mesa 25.2.8-0ubuntu0.24.04.1), renderer: Mesa Intel(R) UHD Graphics (CML GT2)
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.


### Real-Time Monitoring

**Note:** Running this in a notebook works best on a **local Jupyter installation**. 

- Press **'q'** in the camera window to stop the monitoring loop.

In [6]:
cap = cv2.VideoCapture(0)

while cap.isOpened():
    success, frame = cap.read()
    if not success: break

    frame = cv2.flip(frame, 1)
    h, w, _ = frame.shape
    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    results = face_mesh.process(rgb_frame)

    counts = {'Attentive': 0, 'Distracted': 0, 'Drowsy': 0}

    if results.multi_face_landmarks:
        for idx, face_landmarks in enumerate(results.multi_face_landmarks):
            mp_drawing.draw_landmarks(
                image=frame,
                landmark_list=face_landmarks,
                connections=mp_face_mesh.FACEMESH_CONTOURS,
                landmark_drawing_spec=drawing_spec,
                connection_drawing_spec=drawing_spec)

            ear = (calculate_ear(face_landmarks, RIGHT_EYE, w, h) + calculate_ear(face_landmarks, LEFT_EYE, w, h)) / 2
            mar = calculate_mar(face_landmarks, MOUTH, w, h)
            gaze_x = (calculate_gaze(face_landmarks, LEFT_EYE, LEFT_IRIS, w, h) + calculate_gaze(face_landmarks, RIGHT_EYE, RIGHT_IRIS, w, h)) / 2
            
            nose_x, l_eye_x, r_eye_x = face_landmarks.landmark[1].x, face_landmarks.landmark[33].x, face_landmarks.landmark[263].x
            face_width = abs(r_eye_x - l_eye_x)
            head_turn = (nose_x - (l_eye_x + r_eye_x) / 2) / face_width if face_width > 0 else 0

            bx, by, bw, bh = get_face_bbox(face_landmarks, w, h)
            face_crop = frame[max(0, by):min(h, by+bh), max(0, bx):min(w, bx+bw)]
            
            status, color = "ATTENTIVE", (0, 255, 0)
            cnn_hint = ""
            
            if face_crop.size > 0:
                face_rgb = cv2.cvtColor(face_crop, cv2.COLOR_BGR2RGB)
                face_resized = cv2.resize(face_rgb, Config.IMG_SIZE)
                face_input = np.expand_dims(face_resized / 255.0, axis=0)
                raw_pred = model.predict(face_input, verbose=0)[0][0]
                pred = smoother.smooth(idx, raw_pred)
                cnn_hint = f" (CNN: {pred:.2f})"
                
                if mar > Config.MAR_THRESHOLD: 
                    status, color = "YAWNING", (255, 255, 0)
                    counts['Distracted'] += 1
                elif ear < Config.EAR_THRESHOLD: 
                    status, color = "DROWSY", (0, 165, 255)
                    counts['Drowsy'] += 1
                elif abs(head_turn) > 0.4: 
                    status, color = "NOT FOCUS (HEAD)", (0, 0, 255)
                    counts['Distracted'] += 1
                elif abs(gaze_x) > 0.25: 
                    status, color = "LOOKING AWAY", (255, 0, 255)
                    counts['Distracted'] += 1
                else:
                    status, color = "ATTENTIVE", (0, 255, 0)
                    counts['Attentive'] += 1

                cv2.rectangle(frame, (bx, by), (bx+bw, by+bh), color, 2)
                cv2.putText(frame, f"{status}{cnn_hint}", (bx, by-10), Config.FONT, 0.5, color, 2)
                cv2.putText(frame, f"E:{ear:.2f} G:{gaze_x:.2f} H:{head_turn:.2f} M:{mar:.2f}", (bx, by+bh+15), Config.FONT, 0.4, (255, 255, 255), 1)

    overlay = frame.copy()
    cv2.rectangle(overlay, (0, 0), (250, 50), (0, 0, 0), -1)
    frame = cv2.addWeighted(overlay, 0.6, frame, 0.4, 0)
    total = sum(counts.values())
    cv2.putText(frame, f"Total Students: {total}", (10, 30), Config.FONT, 0.7, (255, 255, 255), 2)

    cv2.imshow('Classroom Attention Monitor (MediaPipe)', frame)
    if cv2.waitKey(1) & 0xFF == ord('q'): break

cap.release()
cv2.destroyAllWindows()